# A2.3 · Shadow Autonomy

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.2 · The bootstrap problem](https://spbreed.github.io/cyber-commons/lessons/A2.2.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, SPIRE |
| Open-weight models | Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**Shadow autonomy** is what you get when an agent acts using a human's identity.

It is not a bug anyone wrote. It is the path of least resistance: the agent
needs permissions, the human already has them, handing over the human's token
takes ten minutes and requesting a properly scoped agent identity takes three
weeks. Everyone involved is being reasonable.

The result is that the agent becomes indistinguishable from the person, in every
system that matters:

- The audit log names the human for actions they never saw.
- Anomaly detection tuned to human behaviour sees a human doing 400 things a
  minute and either alerts on everything or gets retuned until it alerts on
  nothing.
- Incident containment aims at the human's account, which does not stop the
  agent if it holds a copy of the token.
- The human is accountable, in the formal sense, for decisions made by a model
  they cannot inspect.

The name is deliberate: the autonomy is real, and it is invisible to every
control you have.

## 2 · Demo — an ordinary Tuesday, correctly recorded

First, what a *properly* attributed session looks like, so the broken one is recognisable by contrast.

In [ ]:
import time
from dataclasses import dataclass, field

@dataclass
class LogLine:
    ts: float
    logged_actor: str      # what the audit log says
    real_actor: str        # what actually happened
    action: str
    target: str = ""

def render(lines, truth=False):
    base = lines[0].ts
    out = [f"{'t+s':>5}  {'actor':16s}{'action':16s}target"]
    for ln in sorted(lines, key=lambda x: x.ts):
        who = ln.real_actor if truth else ln.logged_actor
        out.append(f"{ln.ts-base:>5.0f}  {who:16s}{ln.action:16s}{ln.target}")
    return "\n".join(out)

t0 = time.time()
proper = [
    LogLine(t0,      "dana@corp",    "dana@corp",    "login",       "console"),
    LogLine(t0+30,   "dana@corp",    "dana@corp",    "assign_task", "finding-4471"),
    LogLine(t0+31,   "triage-agent", "triage-agent", "read_source", "src/auth.py"),
    LogLine(t0+33,   "triage-agent", "triage-agent", "open_pr",     "pr/8812"),
]
print("PROPERLY ATTRIBUTED — agent has its own identity")
print(render(proper))

## 3 · Where it breaks — the same Tuesday, one shortcut

Now the agent is handed Dana's token. Nothing else changes: same actions, same times, same outcome. Only the identity is different.

In [ ]:
shadow = [
    LogLine(t0,      "dana@corp", "dana@corp",    "login",        "console"),
    LogLine(t0+30,   "dana@corp", "dana@corp",    "assign_task",  "finding-4471"),
    LogLine(t0+31,   "dana@corp", "triage-agent", "read_source",  "src/auth.py"),
    LogLine(t0+33,   "dana@corp", "triage-agent", "open_pr",      "pr/8812"),
    LogLine(t0+34,   "dana@corp", "triage-agent", "merge_pr",     "pr/8812"),
    LogLine(t0+36,   "dana@corp", "triage-agent", "deploy",       "prod"),
]
print("WHAT THE RESPONDER SEES")
print(render(shadow))
print("\nWHAT ACTUALLY HAPPENED")
print(render(shadow, truth=True))

def reconstruct(lines):
    logged = {l.logged_actor for l in lines}
    real   = {l.real_actor for l in lines}
    wrong  = [l for l in lines if l.logged_actor != l.real_actor]
    return {"actors_in_logs": sorted(logged), "actors_in_reality": sorted(real),
            "misattributed_lines": len(wrong),
            "hidden_actors": sorted(real - logged),
            "attribution": "sound" if not wrong else "BROKEN"}

r = reconstruct(shadow)
for k, v in r.items():
    print(f"{k:22s} {v}")

## 4 · The three controls that misfire

This is the part worth internalising: shadow autonomy does not merely make logs untidy. It **breaks controls you are relying on**, silently.

In [ ]:
# --- control 1: behavioural anomaly detection -------------------------
def human_baseline_alert(lines, actor):
    acts = [l for l in lines if l.logged_actor == actor]
    if len(acts) < 2: return None
    span = max(l.ts for l in acts) - min(l.ts for l in acts)
    rate = len(acts) / max(span, 1e-9) * 60
    return (f"{actor}: {rate:.0f} actions/min — "
            f"{'IMPOSSIBLE for a human, alert' if rate > 30 else 'normal'}")

print("control 1 — anomaly detection tuned for humans")
print("   ", human_baseline_alert(shadow, "dana@corp"))
print("    → this alert fires on every agent-assisted session, so it gets tuned")
print("      down or disabled within a week. Then it never fires again.")

# --- control 2: incident containment ---------------------------------
print("\ncontrol 2 — containment")
def contain(disable_account, lines):
    stopped = {l.real_actor for l in lines if l.logged_actor == disable_account
               and l.real_actor == disable_account}
    still_running = {l.real_actor for l in lines} - stopped
    return sorted(still_running)
print(f"    disable dana@corp → still running: {contain('dana@corp', shadow)}")
print("    → the agent holds a copy of the token. Disabling the human's login")
print("      does not invalidate a bearer token already issued.")

# --- control 3: accountability ---------------------------------------
print("\ncontrol 3 — accountability")
print("    formal record: dana@corp deployed to prod at t+36s")
print("    reality:       dana was in a meeting; a model chose to deploy")
print("    → she is accountable for a decision she could not have reviewed.")

## 5 · The fix, and why it is A2.5's job

The fix is not "log harder". You cannot recover the acting identity afterwards, because it was never transmitted — the resource server saw Dana's token and recorded exactly what it was given.

The fix is that the agent must present a token that names *both*. That is on-behalf-of, and it is A2.5. What this lesson establishes is that the alternative is not merely untidy: it disables anomaly detection, containment and accountability at the same time.

In [ ]:
def containment_options(has_agent_identity):
    if has_agent_identity:
        return ["revoke the agent identity (agent stops, human keeps working)",
                "revoke the human (agent's delegated tokens die with the chain)",
                "revoke one agent in a fleet, leaving the others up"]
    return ["disable the human's account (agent may continue on a live token)",
            "rotate the shared credential (breaks every consumer at once)",
            "…that is the complete list"]

for label, flag in (("with agent identity", True), ("shadow autonomy", False)):
    print(f"{label}:")
    for o in containment_options(flag):
        print(f"   · {o}")
    print()

## What you just proved

The properly-attributed timeline names `triage-agent` for its own actions. The shadow timeline is identical except that all four agent actions are logged as `dana@corp` — attribution BROKEN, 4 misattributed lines, `triage-agent` hidden. The anomaly rule computes an impossible human rate, containment shows the agent still running after the account is disabled, and the containment-options list collapses to two bad choices.

## Your turn

Search your audit logs for a human account performing more than 30 actions in a minute. Every hit is either an incident or shadow autonomy, and you will not be able to tell which from the log alone — which is the point.

---

**Next → [A2.4 · The NHI governance gap](https://spbreed.github.io/cyber-commons/lessons/A2.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*